In [1]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/iqoo-hackathon/iqoo_food_dataset.zip" -d /content/dataset

!ls /content/dataset

Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/iqoo-hackathon/iqoo_food_dataset.zip, /content/drive/MyDrive/iqoo-hackathon/iqoo_food_dataset.zip.zip or /content/drive/MyDrive/iqoo-hackathon/iqoo_food_dataset.zip.ZIP.
ls: cannot access '/content/dataset': No such file or directory


In [2]:
!find /content/drive/MyDrive -iname "*iqoo*"

/content/drive/MyDrive/iqoo-hackathon indian-food-dataset.zip
/content/drive/MyDrive/iqoo-hackathon indian-food-dataset.zip/iqoo_food_dataset.zip


In [3]:
!unzip -q "/content/drive/MyDrive/iqoo-hackathon indian-food-dataset.zip/iqoo_food_dataset.zip" -d /content/dataset

!ls /content/dataset

'Food Classification'


In [4]:
!ls "/content/dataset/Food Classification"

burger	     chole_bhature  idli	  kulfi        pakode
butter_naan  dal_makhani    jalebi	  masala_dosa  pav_bhaji
chai	     dhokla	    kaathi_rolls  momos        pizza
chapati      fried_rice     kadai_paneer  paani_puri   samosa


In [5]:
import tensorflow as tf

DATA_DIR = "/content/dataset/Food Classification"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load training set (80% of images)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load validation set (remaining 20%, used only to check accuracy, never trained on)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Save the class names in order — we'll need this later to map
# predictions back to actual food names
class_names = train_ds.class_names
print("Classes found:", class_names)

Found 6269 files belonging to 20 classes.
Using 5016 files for training.
Found 6269 files belonging to 20 classes.
Using 1253 files for validation.
Classes found: ['burger', 'butter_naan', 'chai', 'chapati', 'chole_bhature', 'dal_makhani', 'dhokla', 'fried_rice', 'idli', 'jalebi', 'kaathi_rolls', 'kadai_paneer', 'kulfi', 'masala_dosa', 'momos', 'paani_puri', 'pakode', 'pav_bhaji', 'pizza', 'samosa']


In [6]:
from tensorflow.keras import layers, models

# Load MobileNetV2, pretrained on ImageNet, WITHOUT its original final layer
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze it — don't update its existing knowledge during training
base_model.trainable = False

# Build our full model: base MobileNet + our own new classification head
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),  # normalize pixels 0-1
    base_model,
    layers.GlobalAveragePooling2D(),   # simplifies MobileNet's output into a compact form
    layers.Dense(128, activation='relu'),  # a small learning layer
    layers.Dropout(0.2),               # helps prevent overfitting
    layers.Dense(len(class_names), activation='softmax')  # final output: 20 food classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,424,532 (9.25 MB)

 Trainable params: 166,548 (650.58 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - accuracy: 0.6156 - loss: 1.3366 - val_accuracy: 0.7757 - val_loss: 0.7945
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - accuracy: 0.8048 - loss: 0.6633 - val_accuracy: 0.8021 - val_loss: 0.6996
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 329s 2s/step - accuracy: 0.8579 - loss: 0.4827 - val_accuracy: 0.8140 - val_loss: 0.6793
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.8915 - loss: 0.3745 - val_accuracy: 0.8180 - val_loss: 0.6540
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 327s 2s/step - accuracy: 0.9177 - loss: 0.2779 - val_accuracy: 0.8140 - val_loss: 0.6741


In [8]:
model.save('/content/drive/MyDrive/iqoo-hackathon/food_classifier_v1.keras')
print("Model saved successfully.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/iqoo-hackathon/food_classifier_v1.keras'

In [9]:
model.save('/content/drive/MyDrive/iqoo-hackathon indian-food-dataset.zip/food_classifier_v1.keras')
print("Model saved successfully.")


Model saved successfully.
